In [1]:
CODE   = "/kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline"
BASE   = "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching"
CACHE  = f"{BASE}/cache"
LABELS = BASE


In [2]:
import subprocess, os
base = f"python {CODE}/train.py --cache_dir {CACHE} --labels {LABELS}/labels_soft.csv --epochs 12 --pretrained --bs 4 --accum 2 --workers 2 --train_slices 16"

def run(gpu, folds, out):
    return subprocess.Popen(
        f"{base} --only_folds {' '.join(map(str, folds))} --out /kaggle/working/{out}",
        shell=True, env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)},
        stdout=open(f"/kaggle/working/{out}.log", "w"), stderr=subprocess.STDOUT)

p0 = run(0, [0, 1, 2], "models_g0")
p1 = run(1, [3, 4], "models_g1")
p0.wait(); p1.wait()

0

In [3]:
import pandas as pd, numpy as np
a = pd.read_csv("/kaggle/working/models_g0/oof.csv")
b = pd.read_csv("/kaggle/working/models_g1/oof.csv")
pc = [c for c in a.columns if c.startswith("p_")]
m = a.copy()
m[pc] = a[pc].fillna(b[pc])
print(m[pc].isna().any(axis=1).sum(), "rows missing predictions")  # expect 0
m.to_csv("/kaggle/working/oof_all.csv", index=False)

0 rows missing predictions
